#### **Random Forest**

- 배깅 방식을 활용
- 데이터 샘플링 + 다수결/평균 사용
- 의사결정나무 사용이 고정 (하드코딩 되어있음)

- 각 노드에서 분할할 때마다 모든 feature에서 일부만 무작위 선택 → tree간의 상관성을 낮춰준다 → 다양성 증가
- 여러 결정 트리를 서로 다르게 학습시킨 뒤 다수결(분류), 평균(회귀)으로 예측하는 배깅 기반의 앙상블
- 붓스트랩(중복 데이터를 허용)과 특성의 무작위 선택을 이용하여 모델간의 상관성을 낮춤으로써 성능과 안정성을 높이는 방법

- 고차원 데이터에서 안정성이 굉장히 높다
- 실무에서 많이 사용되는 알고리즘

- **parameter**
    
    - `n_estimators`
        - default : 100
        - 모델의 개수를 지정
    
    - `criterion`
        - 분류 : gini(default), entropy, log_loss
        - 회귀 : squared_error(default), absolute_error, friedman_mse, poisson
    
    - `max_depth`
        - default : None
        - 트리의 최대 깊이
    
    - `bootstrap`
        - default : True
        - 트리 학습 데이터에 부트스트랩을 사용할 것인가
    
    - `max_features`
        - 분류 : 'sqrt' (feature 개수의 루트 값)
        - 회귀 : 1.0 (모든 feature를 사용)
        - 가능한 값 : int(0.0 ~ 1.0), 'sqrt', 'log2'
    
    - `min_sample_leaf`
        - default : 1
        - leaf node의 최소 샘플의 수
        - 값이 큰 경우, leaf가 너무 세분화되어 과적합되는 것을 방지
    
    - `max_leaf_node`
        - default : None
        - 트리 전체에서 leaf node의 수를 제한
        - 너무 많은 leaf가 발생 → 과적합
        - 너무 적은 leaf가 발생 → 모델의 단순화
    
    - `oob_score`
        - default : False
        - 붓스트랩 사용 시 빠진 샘플들을 이용하여 검증 점수를 출력

    - `max_samples`
        - default : None
        - 붓스트랩이 True인 경우, 각 트리가 사용할 샘플의 수(비율)

    - `max_weight_fraction_leaf`
        - default : 0.0
        - 샘플 가중치의 합이 전체 데이터에서 차지하는 비율
        - 불균형 데이터셋에서 유용
    
    - `class_weight`
        - default : None
        - 데이터의 불균형에서 각각의 가중치를 부여할 수 있는 매개변수
        - 'balanced' : 소수의 데이터에 가중치를 추가적으로 부여하여데이터 불균형 문제를 완화
        - 'balanced_subsample' : 각 트리별로 붓스트랩 데이터의 비율로 가중치를 부여
        - dict 형태로 클래스별 가중치를 고정값으로 부여
    

- **속성**

    - `estimators_`
        - 결정 트리들의 목록
    
    - `feature_importances_`
        - feature들의 중요도
    
    - `oob_score`
        - OOB를 이용한 검증 함수
        - parameter `oob_score`가 True일 때만 사용 가능
    
    - `oob_decision_function`
        - 분류 모델 사용 가능
        - OOB 데이터들의 확률 / 결정 함수
    
    - `oob_prediction_`
        - 회귀 모델 사용 가능
        - OOB 예측값
    
    - `n_feature_in_` / `feature_names_in`
        - 입력되는 feature의 개수
        - 입력되는 feature들의 이름 목록


- **메서드**

    - `fit(x, y)`
        - 모델에 학습
    
    - `predict(x)`
        - 모델을 통한 예측
    
    - `score(x, y)`
        - 분류에서는 정확도, 회귀에서는 r2-score
    
    - `apply(x)`
        - 각 샘플이 각 트리에서 도달하는 leaf의 인덱스를 반환
    
    - `decision_path(x)`
        - 트리 내의 경로

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [2]:
body = pd.read_csv('../data/bodyPerformance.csv')
body.head()

,age,gender,height_cm,weight_kg,body fat_%,diastolic,systolic,gripForce,sit and bend forward_cm,sit-ups counts,broad jump_cm,class
0,27.0,M,172.3,75.24,21.3,80.0,130.0,54.9,18.4,60.0,217.0,C
1,25.0,M,165.0,55.80,15.7,77.0,126.0,36.4,16.3,53.0,229.0,A
2,31.0,M,179.6,78.00,20.1,92.0,152.0,44.8,12.0,49.0,181.0,C
3,32.0,M,174.5,71.10,18.4,76.0,147.0,41.4,15.2,53.0,219.0,B
4,28.0,M,173.8,67.70,17.1,70.0,127.0,43.5,27.1,45.0,217.0,B


In [3]:
# gender column, class column의 값을 숫자로 바꿔주기

from sklearn.preprocessing import LabelEncoder

In [4]:
# LabelEncoder → 문자형 데이터들을 숫자형으로 변환

labelencoder = LabelEncoder()

In [5]:
body['class_1'] = labelencoder.fit_transform(body['class'])

In [10]:
body['gender'] = labelencoder.fit_transform(body['gender'])

In [11]:
body.head()

,age,gender,height_cm,weight_kg,body fat_%,diastolic,systolic,gripForce,sit and bend forward_cm,sit-ups counts,broad jump_cm,class,class_1
0,27.0,1,172.3,75.24,21.3,80.0,130.0,54.9,18.4,60.0,217.0,C,2
1,25.0,1,165.0,55.80,15.7,77.0,126.0,36.4,16.3,53.0,229.0,A,0
2,31.0,1,179.6,78.00,20.1,92.0,152.0,44.8,12.0,49.0,181.0,C,2
3,32.0,1,174.5,71.10,18.4,76.0,147.0,41.4,15.2,53.0,219.0,B,1
4,28.0,1,173.8,67.70,17.1,70.0,127.0,43.5,27.1,45.0,217.0,B,1


In [16]:
X = body.drop(['class', 'class_1'], axis=1)
y = body['class_1']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42, stratify = y
)

In [17]:
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [18]:
pred = clf.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.72      0.88      0.79      1004
           1       0.63      0.59      0.61      1004
           2       0.72      0.66      0.69      1005
           3       0.90      0.83      0.86      1005

    accuracy                           0.74      4018
   macro avg       0.74      0.74      0.74      4018
weighted avg       0.74      0.74      0.74      4018



1. 모델의 개수를 증가
2. 각 트리에서 사용되는 샘플을 70%로 제한
3. `min_sample_leaf`를 기본값 1에서 4로 변경

In [22]:
clf2 = RandomForestClassifier(
    n_estimators = 200,
    max_samples = 1.0,
    min_samples_leaf = 1,
    max_features = 6
)

clf2.fit(X_train, y_train)
pred2 = clf2.predict(X_test)

print(classification_report(y_test, pred2))

              precision    recall  f1-score   support

           0       0.72      0.88      0.79      1004
           1       0.63      0.63      0.63      1004
           2       0.75      0.67      0.71      1005
           3       0.92      0.82      0.87      1005

    accuracy                           0.75      4018
   macro avg       0.75      0.75      0.75      4018
weighted avg       0.75      0.75      0.75      4018



**new dataset**

In [48]:
credit = pd.read_csv('../data/credit_final.csv')
credit.head(3)

,credit.rating,account.balance,credit.duration.months,previous.credit.payment.status,credit.purpose,credit.amount,savings,employment.duration,installment.rate,marital.status,...,residence.duration,current.assets,age,other.credits,apartment.type,bank.credits,occupation,dependents,telephone,foreign.worker
0,1,1,18,3,2,1049,1,1,4,1,...,4,2,21,2,1,1,3,1,1,1
1,1,1,9,3,4,2799,1,2,2,3,...,2,1,36,2,1,2,3,2,1,1
2,1,2,12,2,4,841,2,3,2,1,...,4,1,23,2,1,1,2,1,1,1


In [49]:
credit.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   credit.rating                   1000 non-null   int64
 1   account.balance                 1000 non-null   int64
 2   credit.duration.months          1000 non-null   int64
 3   previous.credit.payment.status  1000 non-null   int64
 4   credit.purpose                  1000 non-null   int64
 5   credit.amount                   1000 non-null   int64
 6   savings                         1000 non-null   int64
 7   employment.duration             1000 non-null   int64
 8   installment.rate                1000 non-null   int64
 9   marital.status                  1000 non-null   int64
 10  guarantor                       1000 non-null   int64
 11  residence.duration              1000 non-null   int64
 12  current.assets                  1000 non-null   int64
 13  age            

In [50]:
credit.describe()

,credit.rating,account.balance,credit.duration.months,previous.credit.payment.status,credit.purpose,credit.amount,savings,employment.duration,installment.rate,marital.status,...,residence.duration,current.assets,age,other.credits,apartment.type,bank.credits,occupation,dependents,telephone,foreign.worker
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.00000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.700000,2.183000,20.903000,2.292000,2.965000,3271.24800,1.874000,2.44600,2.973000,2.372000,...,2.845000,2.358000,35.54200,1.814000,1.928000,1.367000,2.904000,1.155000,1.404000,1.037000
std,0.458487,0.835589,12.058814,0.620581,0.971967,2822.75176,1.196476,1.10558,1.118715,1.067125,...,1.103718,1.050209,11.35267,0.389301,0.530186,0.482228,0.653614,0.362086,0.490943,0.188856
min,0.000000,1.000000,4.000000,1.000000,1.000000,250.00000,1.000000,1.00000,1.000000,1.000000,...,1.000000,1.000000,19.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,1.000000,12.000000,2.000000,2.000000,1365.50000,1.000000,2.00000,2.000000,1.000000,...,2.000000,1.000000,27.00000,2.000000,2.000000,1.000000,3.000000,1.000000,1.000000,1.000000
50%,1.000000,2.000000,18.000000,2.000000,3.000000,2319.50000,1.000000,2.00000,3.000000,3.000000,...,3.000000,2.000000,33.00000,2.000000,2.000000,1.000000,3.000000,1.000000,1.000000,1.000000
75%,1.000000,3.000000,24.000000,3.000000,4.000000,3972.25000,3.000000,4.00000,4.000000,3.000000,...,4.000000,3.000000,42.00000,2.000000,2.000000,2.000000,3.000000,1.000000,2.000000,1.000000
max,1.000000,3.000000,72.000000,3.000000,4.000000,18424.00000,4.000000,4.00000,4.000000,4.000000,...,4.000000,4.000000,75.00000,2.000000,3.000000,2.000000,4.000000,2.000000,2.000000,2.000000


##### 연습 문제

1. credit에서 독립 변수와 종속 변수로 분할
2. train, test 셋으로 데이터를 분할 (8:2)
3. RF 모델을 생성(기본값)하여 모델의 성능을 평가

4. 더미 변수 생성 (credit.purpose, marital.status, apartment.type, occupation)
5. 데이터의 균형이 맞는가?
    - 불균형하다면 샘플링 기법을 사용하여 데이터 균형을 맞춰준다.
        - 오버 샘플링: SMOTE
        - train data 샘플링, test 데이터는 샘플링 X
6. 기본 RF 모델을 이용하여 성능 평가

In [98]:
# 1)
X = credit.drop('credit.rating', axis=1)
y = credit['credit.rating']

In [99]:
# 2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify=y
)

In [100]:
# 3)
clf = RandomForestClassifier()
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.74      0.58      0.65        60
           1       0.84      0.91      0.87       140

    accuracy                           0.81       200
   macro avg       0.79      0.75      0.76       200
weighted avg       0.81      0.81      0.81       200



In [101]:
# 4)
credit2 = pd.get_dummies(credit, columns=['credit.purpose', 'apartment.type', 'occupation', 'marital.status'], drop_first= True)

X = credit2.drop('credit.rating', axis=1)
y = credit2['credit.rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify=y
)

In [102]:
# 5)
credit2['credit.rating'].value_counts()

credit.rating
1    700
0    300
Name: count, dtype: int64

In [103]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy=1)

In [104]:
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
y_train_sm.value_counts()

credit.rating
1    560
0    560
Name: count, dtype: int64

In [105]:
# 6)

clf2 = RandomForestClassifier()
clf2.fit(X_train_sm, y_train_sm)
pred2 = clf2.predict(X_test)
print(classification_report(y_test, pred2))

              precision    recall  f1-score   support

           0       0.62      0.60      0.61        60
           1       0.83      0.84      0.84       140

    accuracy                           0.77       200
   macro avg       0.73      0.72      0.72       200
weighted avg       0.77      0.77      0.77       200



In [106]:
print(classification_report(y_test, pred))
print()
print(classification_report(y_test, pred2))

              precision    recall  f1-score   support

           0       0.74      0.58      0.65        60
           1       0.84      0.91      0.87       140

    accuracy                           0.81       200
   macro avg       0.79      0.75      0.76       200
weighted avg       0.81      0.81      0.81       200


              precision    recall  f1-score   support

           0       0.62      0.60      0.61        60
           1       0.83      0.84      0.84       140

    accuracy                           0.77       200
   macro avg       0.73      0.72      0.72       200
weighted avg       0.77      0.77      0.77       200



In [158]:
# 추가 1)

clf3 = RandomForestClassifier(n_estimators=50)
clf3.fit(X_train_sm, y_train_sm)
pred3 = clf3.predict(X_test)
print(classification_report(y_test, pred3))

              precision    recall  f1-score   support

           0       0.63      0.62      0.62        60
           1       0.84      0.84      0.84       140

    accuracy                           0.78       200
   macro avg       0.73      0.73      0.73       200
weighted avg       0.77      0.78      0.77       200



In [159]:
# 모음

clf = RandomForestClassifier()
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print(classification_report(y_test, pred))
print()

clf2 = RandomForestClassifier()
clf2.fit(X_train_sm, y_train_sm)
pred2 = clf2.predict(X_test)
print(classification_report(y_test, pred2))
print()

clf3 = RandomForestClassifier(n_estimators=500)
clf3.fit(X_train_sm, y_train_sm)
pred3 = clf3.predict(X_test)
print(classification_report(y_test, pred3))

              precision    recall  f1-score   support

           0       0.75      0.45      0.56        60
           1       0.80      0.94      0.86       140

    accuracy                           0.79       200
   macro avg       0.77      0.69      0.71       200
weighted avg       0.78      0.79      0.77       200


              precision    recall  f1-score   support

           0       0.54      0.53      0.54        60
           1       0.80      0.81      0.80       140

    accuracy                           0.72       200
   macro avg       0.67      0.67      0.67       200
weighted avg       0.72      0.72      0.72       200


              precision    recall  f1-score   support

           0       0.59      0.58      0.59        60
           1       0.82      0.83      0.83       140

    accuracy                           0.76       200
   macro avg       0.71      0.71      0.71       200
weighted avg       0.75      0.76      0.75       200



In [160]:
feature_df = pd.DataFrame(zip(X_train_sm.columns, clf3.feature_importances_),
             columns=['feature_name', 'importance'])
feature_df.sort_values('importance', ascending=False).head(5)

,feature_name,importance
0,account.balance,0.166811
3,credit.amount,0.097583
1,credit.duration.months,0.086370
10,age,0.075990
4,savings,0.062020


**RandomForestRegressor**

In [161]:
car = pd.read_csv('../data/CarPrice_Assignment.csv')
df = car.select_dtypes('number')
df.drop('car_ID', axis=1, inplace=True)

In [162]:
X = df.drop('price', axis=1)
y = df['price']

In [163]:
reg = RandomForestRegressor(oob_score=True)

In [164]:
reg.fit(X, y)
reg.oob_score_

0.9283194830568189